**NetID:** nr207489

Kaggle competition page: [https://www.kaggle.com/competitions/titanic](https://www.kaggle.com/competitions/titanic)

## Overview

**Task:** Train on the given data to predict whether the passengers in the test dataset will survive.

This is a categorization problem. As such, I will use binary classification with logistic regression.

## Setup and initial analysis

In [370]:
#| output: false
#| code-fold: true
#| code-summary: Initial environment setup

# Better type annotations
from __future__ import annotations

# MAC USERS TAKE NOTE:
# For clearer plots in Jupyter notebooks on Macs, run the following line of code:
%config InlineBackend.figure_format = 'retina'

# Set the base path
from pathlib import Path
BASE_PATH = Path('.').resolve()

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Now let's import the data.

In [371]:
df_train = pd.read_csv(BASE_PATH / 'train.csv').set_index('PassengerId')

df_test = pd.read_csv(BASE_PATH / 'test.csv').set_index('PassengerId')
df_train

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S


**Column definitions**

- Survived: whether the passenger survived
  - 0 = No
  - 1 = Yes
- Pclass: ticket class
  - 1 = 1st
  - 2 = 2nd
  - 3 = 3rd
- Sex: biological sex (silly people of 1912, not separating gender from sex)
- Age: passenger age, in years
- SibSp: # of siblings or spouses aboard the Titanic
- Parch: # of parents or children aboard the titanic
- Ticket: ticket number
- Fare: passenger fare, in 1912 USD
- Cabin: cabin number
- Embarked: Port of embarkation
  - C = Cherbourg
  - Q = Queenstown
  - S = Southhampton

### Preprocessing

We need to use Sex and Embarked, which are both categorical data. Sex (in this data) is binary, so that can be done with a simple binary encoding. Embarked is not, so we'll use one-hot encoding to make it numerical, and thus usable in regression.

In [411]:
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
import copy

# Can't use a simple pipeline here, as we need to select and
# transform specific columns from a DataFrame.
def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    df = copy.deepcopy(df)
    
    # EMBARKED
    
    # https://www.youtube.com/watch?v=rsyrZnZ8J2o
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)\
        .set_output(transform='pandas')  # Tell the encoder to output a DataFrame.
    
    embarked_transformed = encoder.fit_transform(df[['Embarked']])
    df = pd.concat([df, embarked_transformed], axis=1)\
        .drop(columns=['Embarked'])
    
    # SEX
    
    lb = LabelBinarizer()
    sex_transformed = lb.fit_transform(df[['Sex']])
    df[['Sex']] = sex_transformed
    
    return df


df_train_t = encode_categorical(df_train)
df_train_t.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
PassengerId,,,,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,NaN,0.0,0.0,1.0,0.0
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,1.0,0.0,0.0,0.0
3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0.0,0.0,1.0,0.0
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,C123,0.0,0.0,1.0,0.0
5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,NaN,0.0,0.0,1.0,0.0


I noticed that the transformed data has a category for NaN values. Sure enough, there are two passengers which do not have any data for Embarked. I'll simply throw them out. Two missing passengers isn't going to affect the model too much.

In [373]:
embarked_nan = df_train_t.query('Embarked_nan == 1.0')
embarked_nan

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
PassengerId,,,,,,,,,,,,,,
62,1,1,"Icard, Miss. Amelie",0,38.0,0,0,113572,80.0,B28,0.0,0.0,0.0,1.0
830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",0,62.0,0,0,113572,80.0,B28,0.0,0.0,0.0,1.0


In [374]:
df_train_t.drop(
    labels=[i for i in embarked_nan.index],
    axis='index',
    inplace=True
)

I would love to use Cabin as well, as it provides some rough location information for where the passengers *might* have been. However, most passengers are missing this data, so I shouldn't use it, at least right now.

In [375]:
print(f"Qty of passengers missing cabin data: {len(df_train_t.query('Cabin.isnull()'))}")

Qty of passengers missing cabin data: 687


## Model

Alright, let's build the model! I'll use a simple `sklearn.preprocessing.StandardScaler` \[[docs](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)] in an Scikit Learn Pipeline.

In [376]:
# Training data
X_train = df_train_t[[
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()
Y_train = df_train_t[['Survived']].to_numpy()

# Testing data
df_test_t = encode_categorical(df_test)
X_test = df_test_t[[
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()

In [377]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression()),
])

try:
    pipeline.fit(X_train, Y_train)
except Exception as e:
    print(e)

# print(f'Training score: {pipeline.score(X_train, Y_train)}')
# print(f'Test score: {pipeline.score(X_test, Y_test)}')

Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values


### Filling in NaN's

Right. NaN values. Let's see what's causing issues, first in the training set:

In [378]:
df_train_t.isnull().any()

Survived        False
Pclass          False
Name            False
Sex             False
Age              True
SibSp           False
Parch           False
Ticket          False
Fare            False
Cabin            True
Embarked_C      False
Embarked_Q      False
Embarked_S      False
Embarked_nan    False
dtype: bool

It looks like just the Age and Cabin values have NaN's in the training set. We can ignore the Cabin values, as those aren't being used in this model (see earlier note about high NaN count). The Age values are going to be a problem. How many do we have?

In [379]:
len(df_train_t.query("Age.isnull()"))

177

Okay, there are two ways we could go about this. We could either:

1. Use another regression model to insert predicted values using the passengers with known ages, or
2. Simply fill in each missing Age value with the average in the test dataset.

Both have their advantages and disadvantages, but I'm going to attempt the latter, as I predict it'll result in better performance of the main classification model.

I'll use Lasso regression, as I want to quickly filter out input features that aren't correlated to the passenger's age. Our training and testing data will both originate from the larger training dataset, but with all passengers without ages removed.

#### Method 1: Lasso Regression

In [380]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Select all passengers with age data from training dataset
age_train = df_train_t.query("not Age.isnull()")

# Grab the desired features and predictor
age_X_train = age_train[[
    'Pclass', 'Sex', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()
age_Y_train = age_train[['Age']].to_numpy()

# Do the same, but for the test dataset
age_test = df_test_t.query("not Age.isnull()")

Before we continue, we need to do a bit more interpolation. Mr. Thomas Storey from the test dataset has a null Fare value, which I would very much like to keep in this regression, so we'll fill it in with the average fare value.

In [381]:
age_test.query('Fare.isnull()')

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1044,3,"Storey, Mr. Thomas",1,60.5,0,0,3701,NaN,NaN,0.0,0.0,1.0


In [382]:
fare_null = age_test.query('Fare.isnull()')
fare_avg = np.mean(age_test.query('not Fare.isnull()')[['Fare']].to_numpy())

for idx in fare_null.index:
    age_test.at[idx, 'Fare'] = fare_avg

Now we can continue training the inner model.

In [383]:
age_X_test = age_test[[
    'Pclass', 'Sex', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()
age_Y_test = age_test[['Age']].to_numpy()


age_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3)),
    ('model', Lasso(alpha=0.3, fit_intercept=True)),
])

age_pipeline.fit(age_X_train, age_Y_train)

age_train_MSE = mean_squared_error(age_Y_train, age_pipeline.predict(age_X_train))
age_test_MSE = mean_squared_error(age_Y_test, age_pipeline.predict(age_X_test))

print(f'Train score: {age_pipeline.score(age_X_train, age_Y_train):.6f}, MSE = {age_train_MSE:.6f}')
print(f'Test score: {age_pipeline.score(age_X_test, age_Y_test):.6f}, MSE = {age_test_MSE:.6f}')

Train score: 0.326259, MSE = 141.317209
Test score: 0.245949, MSE = 151.187856


#### Method 2: Simple Average

That's... not great, so let's try the average method and see how it does.

In [384]:
age_avg = np.mean(age_Y)
print(f'Average age: {age_avg:.2f}')
print(f'Average MSE = {mean_squared_error(age_Y, [age_avg] * len(age_Y)):.6f}')

Average age: 29.70
Average MSE = 210.723580


#### Evaluation and Use

The average age method *is* less accurate, which makes sense, so I'll use the
Lasso-predicted data instead. I'll use it to fill in the missing data in both
the training and testing datasets.

In [385]:
def fill_in_nan_ages(df: pd.DataFrame) -> pd.DataFrame:
    missing_age = df.query("Age.isnull()")[[
        'Pclass', 'Sex', 'SibSp', 'Parch', 'Fare',
        'Embarked_C', 'Embarked_Q', 'Embarked_S'
    ]].to_numpy()
    pred = age_pipeline.predict(missing_age)

    df_copy = copy.deepcopy(df)

    for i, v in zip(df.query("Age.isnull()").index, pred):
        df_copy.at[i, 'Age'] = v

    return df_copy

In [386]:
df_train_ta = fill_in_nan_ages(df_train_t)
df_train_ta.isnull().any()

Survived        False
Pclass          False
Name            False
Sex             False
Age             False
SibSp           False
Parch           False
Ticket          False
Fare            False
Cabin            True
Embarked_C      False
Embarked_Q      False
Embarked_S      False
Embarked_nan    False
dtype: bool

In [387]:
df_test_ta = fill_in_nan_ages(df_test_t)
df_test_ta.isnull().any()

Pclass        False
Name          False
Sex           False
Age           False
SibSp         False
Parch         False
Ticket        False
Fare           True
Cabin          True
Embarked_C    False
Embarked_Q    False
Embarked_S    False
dtype: bool

We are now NaN-free! Back to the task at hand: predicting who will die.

## Classification Model (cont'd)

Now that our data lacks NaN's, we can resume training the main logistic regression binary classification model. First, we'll redefine our training and testing datasets from the preprocessed data, then we'll train it.

In [388]:
# Training data
X_train = df_train_ta[[
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()
Y_train = df_train_ta[['Survived']].to_numpy()

In [396]:
# Run the NaN Fare fix from above
fare_null = df_test_ta.query('Fare.isnull()')
fare_avg = np.mean(df_test_ta.query('not Fare.isnull()')[['Fare']].to_numpy())

for idx in fare_null.index:
    df_test_ta.at[idx, 'Fare'] = fare_avg

# Testing data
X_test = df_test_ta[[
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'Embarked_C', 'Embarked_Q', 'Embarked_S'
]].to_numpy()

In [397]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression()),
])

# np.ravel(Y_train) is used to convert from a 2D array
# of len-1 arrays to one 1D array
pipeline.fit(X_train, np.ravel(Y_train))

print(f'Training score: {pipeline.score(X_train, Y_train)}')

Training score: 0.8098987626546682


## Predict!

Let's use the model to predict whether the passengers in the test dataset will survive.

In [398]:
Y_hat = pipeline.predict(X_test)

In [409]:
# df_test_predicted = copy.deepcopy(df_test_ta)
# df_test_predicted.insert(0, 'Survived', Y_hat)
# df_test_predicted.head()
df_predicted = pd.DataFrame(
    data=Y_hat,
    index=df_test_ta.index,
    columns=['Survived'],
)
df_predicted

,Survived
PassengerId,
892,0
893,0
894,0
895,0
896,1
...,...
1305,0
1306,1
1307,0


In [410]:
df_predicted.to_csv(BASE_PATH / 'predictions.csv')

![Kaggle Submission Dialog](kaggle-submission-dialog.jpg)